Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [28]:
!pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 31.9 MB/s eta 0:00:00


Import Packages

In [2]:
import pandas as pd
import xml.etree.ElementTree as ET
import json
import os

from collections import defaultdict

Directory Setup

In [6]:
BASE = "/content/drive/MyDrive/Bioinformatics/"
OUT_DIR = BASE + "outputs/"
VARIANT_DIR = OUT_DIR + "variants/"

PRED = BASE + "outputs/variants/patient1_top20_pathogenic.csv"

ORPHANET_GENE = BASE + "data/external/orphanet/disease_gene_associations.xml"
ORPHANET_PHENO = BASE + "data/external/orphanet/disease_phenotype.xml"

HPO_ASSOC = BASE + "data/external/hpo/phenotype.hpoa"
HPO_JSON = BASE + "data/external/hpo/hp.json"

DIAG_DIR = BASE + "outputs/diagnosis/"
os.makedirs(DIAG_DIR, exist_ok=True)

df_pred = pd.read_csv(PRED)
df_path = df_pred[df_pred["Prediction"] == 1]

ORPHANET gene → disease mapping

In [7]:
tree = ET.parse(ORPHANET_GENE)
root = tree.getroot()

gene_to_disease = {}

for disorder in root.findall(".//Disorder"):
    disease_name = disorder.findtext("Name", default="Unknown Disease")

    for gene in disorder.findall(".//Gene"):
        gene_symbol = gene.findtext("Symbol")

        if gene_symbol:
            gene_to_disease.setdefault(gene_symbol, set()).add(disease_name)

gene_to_disease = {g: sorted(list(v)) for g, v in gene_to_disease.items()}

print("Orphanet genes mapped:", len(gene_to_disease))

Orphanet genes mapped: 4552


ORPHANET disease → HPO mapping

In [8]:
tree_p = ET.parse(ORPHANET_PHENO)
root_p = tree_p.getroot()

disease_to_hpo = {}

for disorder in root_p.findall(".//Disorder"):
    disease_name = disorder.findtext("Name", default="Unknown Disease")
    hpo_list = [
        pheno.findtext("HPOId")
        for pheno in disorder.findall(".//Phenotype")
        if pheno.findtext("HPOId") is not None
    ]
    disease_to_hpo[disease_name] = list(set(hpo_list))

print("Diseases with HPO data:", len(disease_to_hpo))

Diseases with HPO data: 4337


HPO JSON → ID → name

In [9]:
with open(HPO_JSON, "r") as f:
    hp = json.load(f)

id_to_hpo_name = {
    term["id"]: term.get("name", "")
    for term in hp["graphs"][0]["nodes"]
    if "id" in term
}

print("Total HPO terms:", len(id_to_hpo_name))

Total HPO terms: 19986


Build final mapping table

In [10]:
results = []

for _, row in df_path.iterrows():
    gene = row["GeneName"]

    # Orphanet Diseases for the gene
    diseases = gene_to_disease.get(gene, [])

    # Add phenotype list per disease
    disease_entries = []
    for d in diseases:
        hpos = disease_to_hpo.get(d, [])
        disease_entries.append({
            "Disease": d,
            "Phenotypes": [id_to_hpo_name.get(h, h) for h in hpos]
        })

    results.append({
        "CHROM": row["CHROM"],
        "POS": row["POS"],
        "REF": row["REF"],
        "ALT": row["ALT"],
        "GeneName": gene,
        "Effect": row["Effect"],
        "Impact": row["Impact"],
        "Probability": row["Probability"],
        "Diseases": diseases,
        "DiseaseDetails": disease_entries
    })

df_diag = pd.DataFrame(results).sort_values("Probability", ascending=False)
df_diag.head()

,CHROM,POS,REF,ALT,GeneName,Effect,Impact,Probability,Diseases,DiseaseDetails
0,chr13,32338844,TACTC,T,BRCA2,frameshift_variant,HIGH,0.735049,[Breast implant-associated anaplastic large ce...,[{'Disease': 'Breast implant-associated anapla...
1,chr13,32337218,CT,C,BRCA2,frameshift_variant,HIGH,0.733739,[Breast implant-associated anaplastic large ce...,[{'Disease': 'Breast implant-associated anapla...
2,chr13,32339569,CT,C,BRCA2,frameshift_variant,HIGH,0.733711,[Breast implant-associated anaplastic large ce...,[{'Disease': 'Breast implant-associated anapla...
3,chr13,32202452,AAGAC,A,FRY,frameshift_variant,HIGH,0.733393,[],[]
4,chr12,31792140,ATGAG,A,H3F3C,frameshift_variant,HIGH,0.732964,[],[]


SYMPTOM CROSS CHECK

In [20]:
patient_symptom_names = ["breast swelling", "breast Pain", "breast lump", "skin redness", "weight loss", "night sweats"]

print("Symptoms entered:", patient_symptom_names)

Symptoms entered: ['breast swelling', 'breast Pain', 'breast lump', 'skin redness', 'weight loss', 'night sweats']


Check if symptoms exist

In [21]:
if len(patient_symptom_names) == 0:
    print("\nNo symptoms entered — skipping symptom matching step.\n")

    # Create empty result file with header
    df_symptom = pd.DataFrame(columns=[
        "Disease", "Gene", "Variant", "Overlap",
        "MatchFraction", "ML_Probability", "FinalScore"
    ])

    OUT_SYMPTOM = DIAG_DIR + "patient1_symptom_ranked_diseases.csv"
    df_symptom.to_csv(OUT_SYMPTOM, index=False)

    print("Created empty symptom-matching file:", OUT_SYMPTOM)

    # STOP here (do not run matching logic)
    symptom_matching_enabled = False

else:
    symptom_matching_enabled = True

Only convert symptoms if provided

In [23]:
if symptom_matching_enabled:

    patient_hpo_ids = []

    for s in patient_symptom_names:
        key = s.lower().strip()
        if key in id_to_hpo_name:
            patient_hpo_ids.append(id_to_hpo_name[key])
        else:
            print("Symptom not found in HPO:", s)

    print("Patient HPO IDs:", patient_hpo_ids)

Symptom not found in HPO: breast swelling
Symptom not found in HPO: breast Pain
Symptom not found in HPO: breast lump
Symptom not found in HPO: skin redness
Symptom not found in HPO: weight loss
Symptom not found in HPO: night sweats
Patient HPO IDs: []


Score diseases across all pathogenic variants

In [24]:
if symptom_matching_enabled:

    def symptom_overlap(listA, listB):
        return len(set(listA) & set(listB))

    scored = []

    for _, row in df_diag.iterrows():
        gene = row["GeneName"]

        for entry in row["DiseaseDetails"]:
            disease = entry["Disease"]
            phenotype_names = entry["Phenotypes"]

            # Convert disease phenotype names → HPO IDs
            disease_hpo = []
            for name in phenotype_names:
                key = name.lower()
                if key in id_to_hpo_name:
                    disease_hpo.append(id_to_hpo_name[key])

            overlap = symptom_overlap(patient_hpo_ids, disease_hpo)
            match_fraction = overlap / max(1, len(patient_hpo_ids))
            score = match_fraction * row["Probability"]

            scored.append({
                "Disease": disease,
                "Gene": gene,
                "Variant": f"{row['CHROM']}:{row['POS']} {row['REF']}>{row['ALT']}",
                "Overlap": overlap,
                "MatchFraction": match_fraction,
                "ML_Probability": row["Probability"],
                "FinalScore": score
            })

    df_symptom = pd.DataFrame(scored)
    df_symptom = df_symptom.sort_values("FinalScore", ascending=False)

Save to Drive

In [25]:
OUTPUT = DIAG_DIR + "patient1_disease_mapping.csv"
df_diag.to_csv(OUTPUT, index=False)

print("Disease mapping saved:", OUTPUT)

OUT_SYMPTOM = DIAG_DIR + "patient1_symptom_ranked_diseases.csv"
df_symptom.to_csv(OUT_SYMPTOM, index=False)

print("Saved symptom-ranked diseases to:", OUT_SYMPTOM)

Disease mapping saved: /content/drive/MyDrive/Bioinformatics/outputs/diagnosis/patient1_disease_mapping.csv
Saved symptom-ranked diseases to: /content/drive/MyDrive/Bioinformatics/outputs/diagnosis/patient1_symptom_ranked_diseases.csv


FINAL DISEASE LIKELIHOOD SCORING

In [26]:
df_final = df_symptom.copy()

# Variant Count Weight
variant_weights = (
    df_final.groupby("Disease")["Variant"]
    .count()
    .to_dict()
)

# Impact Weighting
impact_map = {
    "HIGH": 1.0,
    "MODERATE": 0.7,
    "LOW": 0.3,
    "MODIFIER": 0.1
}

impact_weight = {}
for _, row in df_diag.iterrows():
    for entry in row["DiseaseDetails"]:
        impact_weight[entry["Disease"]] = impact_map.get(row["Impact"], 0.3)

# Combine weights
scores = []
alpha, beta, gamma, delta = 0.5, 0.35, 0.10, 0.05

for _, row in df_final.iterrows():
    dis = row["Disease"]

    score = (
        alpha * row["ML_Probability"]
        + beta * row["MatchFraction"]
        + gamma * (variant_weights.get(dis, 1) / 5)
        + delta * impact_weight.get(dis, 0.3)
    )

    scores.append(score)

df_final["DiseaseLikelihoodScore"] = scores
df_final = df_final.sort_values("DiseaseLikelihoodScore", ascending=False)
df_final.head(20)

,Disease,Gene,Variant,Overlap,MatchFraction,ML_Probability,FinalScore,DiseaseLikelihoodScore
0,Breast implant-associated anaplastic large cel...,BRCA2,chr13:32338844 TACTC>T,0,0.0,0.735049,0.0,0.477525
1,Cholangiocarcinoma,BRCA2,chr13:32338844 TACTC>T,0,0.0,0.735049,0.0,0.477525
2,Chordoma,BRCA2,chr13:32338844 TACTC>T,0,0.0,0.735049,0.0,0.477525
3,Familial colorectal cancer Type X,BRCA2,chr13:32338844 TACTC>T,0,0.0,0.735049,0.0,0.477525
4,Familial pancreatic carcinoma,BRCA2,chr13:32338844 TACTC>T,0,0.0,0.735049,0.0,0.477525
5,Familial prostate cancer,BRCA2,chr13:32338844 TACTC>T,0,0.0,0.735049,0.0,0.477525
6,Fanconi anemia,BRCA2,chr13:32338844 TACTC>T,0,0.0,0.735049,0.0,0.477525
7,Hereditary breast and/or ovarian cancer syndrome,BRCA2,chr13:32338844 TACTC>T,0,0.0,0.735049,0.0,0.477525
8,Hereditary breast cancer,BRCA2,chr13:32338844 TACTC>T,0,0.0,0.735049,0.0,0.477525
9,Inflammatory breast cancer,BRCA2,chr13:32338844 TACTC>T,0,0.0,0.735049,0.0,0.477525


GENERATE FULL CLINICAL PDF REPORT

In [29]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import A4

REPORT_PATH = DIAG_DIR + "patient_final_clinical_report.pdf"

styles = getSampleStyleSheet()
report = SimpleDocTemplate(REPORT_PATH, pagesize=A4)
flow = []

# Title
flow.append(Paragraph("<b>Clinical Genomic Diagnosis Report</b>", styles["Title"]))
flow.append(Spacer(1, 20))

# Summary
summary = f"""
<b>Summary</b><br/><br/>
Total Pathogenic Variants: {len(df_path)}<br/>
Symptoms Provided: {', '.join(patient_symptom_names) if len(patient_symptom_names)>0 else 'None'}<br/>
Most Likely Diagnosis: {df_final.iloc[0]['Disease']}<br/>
Responsible Gene: {df_final.iloc[0]['Gene']}<br/>
Likelihood Score: {df_final.iloc[0]['DiseaseLikelihoodScore']:.3f}<br/>
"""
flow.append(Paragraph(summary, styles["Normal"]))
flow.append(Spacer(1, 20))

# Top 5 Diseases Table
top5 = df_final.head(5)[["Disease","Gene","DiseaseLikelihoodScore"]]
flow.append(Paragraph("<b>Top 5 Likely Diagnoses</b>", styles["Heading2"]))
flow.append(Table([top5.columns.tolist()] + top5.values.tolist()))
flow.append(Spacer(1, 20))

# Top Variants
flow.append(Paragraph("<b>Key Pathogenic Variants</b>", styles["Heading2"]))
for _, r in df_path.sort_values("Probability", ascending=False).head(5).iterrows():
    text = f"""
    {r['CHROM']}:{r['POS']} {r['REF']}>{r['ALT']}<br/>
    Gene: {r['GeneName']}<br/>
    Effect: {r['Effect']} | Impact: {r['Impact']}<br/>
    Probability: {r['Probability']:.3f}<br/><br/>
    """
    flow.append(Paragraph(text, styles["Normal"]))
flow.append(Spacer(1, 20))

# Save PDF
report.build(flow)

print("Clinical PDF saved at:", REPORT_PATH)

Clinical PDF saved at: /content/drive/MyDrive/Bioinformatics/outputs/diagnosis/patient_final_clinical_report.pdf
